In [1]:
!pip install imbalance-metrics

In [2]:
!pip install ImbalancedLearningRegression

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.2/77.2 kB 5.0 MB/s eta 0:00:00


In [3]:
!pip install smogn
!pip install resreg

In [4]:
from joblib import Parallel, delayed

import pandas as pd
from sklearn.model_selection import train_test_split, RepeatedKFold, GridSearchCV, KFold
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor
from glob import glob
import numpy as np
import os
import smogn
import resreg
from xgboost import XGBRegressor
import itertools as it

from imbalance_metrics import regression_metrics as rm
import ImbalancedLearningRegression as iblr

import warnings
warnings.filterwarnings('ignore')

In [5]:
def train(regressor, strategy, X, y, c, dataset_name):

  train = np.column_stack((y, X))
  train_output_file = f"train_SG_{dataset_name}.csv"  # Nome único para cada conjunto de treino
  pd.DataFrame(train).to_csv(train_output_file, index=False)
  train = pd.read_csv(train_output_file)

  try:
      train = balance(train, strategy, c)
  except ValueError:
      pass

  X = train.drop([train.columns[0]], axis=1)
  y = train[train.columns[0]]

  model = regressor.fit(X.values, y.values)

  return model

In [6]:
def balance(train, strategy, c):

  if strategy == "GN":
    train = iblr.gn(data = train, y = "0", samp_method=c[0], pert=c[1],  rel_thres = 0.8)
  elif strategy == "RO":
    train = iblr.ro(data = train, y = "0", samp_method=c[0], rel_thres = 0.8)
  elif strategy == "RU":
    train = iblr.random_under(data = train, y = "0", samp_method=c[0], rel_thres = 0.8)
  elif strategy == "SG":
    train =  train.dropna()
    train = smogn.smoter(data = train, y = train.columns[0], samp_method=c[2], k=c[0], pert=c[1], rel_xtrm_type = 'high', rel_thres = 0.8)
    train =  train.dropna()
  elif strategy == "SMT":
    train = iblr.smote(data = train, y = "0", samp_method=c[0], rel_thres = 0.8)
  elif strategy == "WC":
    X_train = train.drop([train.columns[0]], axis = 1)
    y_train  = train[train.columns[0]]
    relevance = resreg.pdf_relevance(y_train)
    X_wercs, y_wercs = resreg.wercs(X_train, y_train, relevance, over=c[0], under=c[1])
    train = pd.DataFrame(np.column_stack((y_wercs, X_wercs)))
  return train

In [7]:
def repeatedKfold(X, y, dataset_name):

  outer = RepeatedKFold(n_splits=10, n_repeats=2, random_state=42)
  inner = KFold(n_splits=2, random_state=42, shuffle=True)

  print(outer)

  all_result = []

  v_pert = np.arange(0, 1.05, 0.05).tolist()
  val = np.arange(0.3, 1, 0.2).tolist()

  strategys = {"SMT":{"C.perc":["balance", "extreme"], "k": [3, 5, 7]},
               "RO":{"C.perc":["balance", "extreme"]},
               "RU":{"C.perc":["balance", "extreme"]},
               "GN":{"C.perc":["balance", "extreme"], "pert": v_pert},
               "SG":{"samp_method":["balance", "extreme"], "k": [3, 5, 7], "pert": v_pert},
               "WC":{"over": val, "under": val}}

  regressors = {
    'BG': BaggingRegressor(),
    'DT': DecisionTreeRegressor(),
    'MLP': MLPRegressor(),
    'RF': RandomForestRegressor(),
    'SVM': SVR(),
    'XG': XGBRegressor()
  }

  all_results_df = pd.DataFrame(columns=['Fold', 'Strategy', 'BestC', 'BestSERA'])

  for strategy in strategys:
      print(strategy)
      data_frame = []
      params = strategys[strategy]
      keys = sorted(params)

      for regressor_name, regressor in regressors.items():
        print(regressor_name)
        for fold, (train_index, test_index) in enumerate(outer.split(X, y)):
            print("outer")
            print("Fold:", fold)
            X_train_outer, X_test_outer = X[train_index], X[test_index]
            y_train_outer, y_test_outer = y[train_index], y[test_index]

            best_sera = float('inf')
            best_c = None

            combinations = it.product(*(params[Name] for Name in keys))

            for c in combinations:
                print(strategy)
                print(c)

                fold_scores = []

                for train_inner_index, val_inner_index in inner.split(X_train_outer, y_train_outer):
                    score_perc = []
                    print("Inner loop")

                    X_train_inner, X_val_inner = X[train_inner_index], X[val_inner_index]
                    y_train_inner, y_val_inner = y[train_inner_index], y[val_inner_index]

                    model_inner = train(regressor, strategy, X_train_inner, y_train_inner, c, dataset_name)
                    y_pred = model_inner.predict(X_val_inner)
                    sera = rm.sera(y_val_inner, y_pred)
                    fold_scores.append(sera)

                avg_sera = np.mean(fold_scores)
                print("Average SERA:", avg_sera)

                if avg_sera < best_sera:
                    best_sera = avg_sera
                    best_c = c
                print(best_c)

            model_outer = train(regressor, strategy, X_train_outer, y_train_outer, best_c, dataset_name)
            y_pred_outer = model_outer.predict(X_test_outer)
            sera_outer = rm.sera(y_test_outer, y_pred_outer)
            print("sera_outer", sera_outer)

            model_name = type(model_outer).__name__

            #test = np.column_stack((test_index, y_test_outer))
            #pd.DataFrame(test).to_csv('Content/'+strategy+'/'+dataset_name+'.csv/'+model_name+'/Test{}_{}_{}.csv'.format(fold, strategy, model_name, index = False))
            pred = np.column_stack((test_index, y_pred_outer))
            pd.DataFrame(pred).to_csv('/Content/'+strategy+'/'+dataset_name+'.csv/'+model_name+'/Pred{}_{}_{}.csv'.format(fold, strategy, model_name, index = False))

In [13]:
wine = pd.read_csv('/content/drive/MyDrive/TCC/dados/wine_quality.csv')
a1 = pd.read_csv('/content/drive/MyDrive/TCC/dados/a1.csv')
a2 = pd.read_csv('/content/drive/MyDrive/TCC/dados/a2.csv')
a3 = pd.read_csv('/content/drive/MyDrive/TCC/dados/a3.csv')
a7 = pd.read_csv('/content/drive/MyDrive/TCC/dados/a7.csv')
acceleration = pd.read_csv('/content/drive/MyDrive/TCC/dados/acceleration.csv')
airfoild = pd.read_csv('/content/drive/MyDrive/TCC/dados/airfoild.csv')
boston = pd.read_csv('/content/drive/MyDrive/TCC/dados/boston.csv')
concreteStrength = pd.read_csv('/content/drive/MyDrive/TCC/dados/concreteStrength.csv')
heat = pd.read_csv('/content/drive/MyDrive/TCC/dados/heat.csv')
sensory = pd.read_csv('/content/drive/MyDrive/TCC/dados/sensory.csv')
analcatdata_apnea3 = pd.read_csv('/content/drive/MyDrive/TCC/dados/analcatdata_apnea3.csv')
available_power = pd.read_csv('/content/drive/MyDrive/TCC/dados/available_power.csv')
cocomo_numeric = pd.read_csv('/content/drive/MyDrive/TCC/dados/cocomo_numeric.csv')
debutanizer = pd.read_csv('/content/drive/MyDrive/TCC/dados/debutanizer.csv')
fuel_consumption_country = pd.read_csv('/content/drive/MyDrive/TCC/dados/fuel_consumption_country.csv')

In [14]:
datasets = {
    "wine": {"data": wine, "target": "target"},
    "a1": {"data": a1, "target": "a1"},
    "a2": {"data": a2, "target": "a2"},
   # "a3": {"data": a3, "target": "a3"},
   # "a7": {"data": a7, "target": "a7"},
    "acceleration": {"data": acceleration, "target": "target"},
    "airfoild": {"data": airfoild, "target": "target"},
    "boston": {"data": boston, "target": "target"},
   # "concreteStrength": {"data": concreteStrength, "target": "target"},
    "heat": {"data": heat, "target": "heat"},
    "analcatdata_apnea3": {"data": analcatdata_apnea3, "target": "target"},
    "available_power": {"data": available_power, "target": "target"},
    #"cocomo_numeric": {"data": cocomo_numeric, "target": "target"},
    #"debutanizer": {"data": debutanizer, "target": "target"},
    "sensory": {"data": sensory, "target": "target"},
    #"fuel_consumption_country": {"data": fuel_consumption_country, "target": "fuel.consumption.country"},
    }

In [ ]:
for dataset_name, info in datasets.items():

    print(f"\n=== Rodando dataset: {dataset_name} ===")

    df = info["data"].copy()
    target = info["target"]

    X = df.drop(columns=[target]).to_numpy()
    y = df[target].to_numpy()

    repeatedKfold(X, y, dataset_name)



=== Rodando dataset: wine ===
RepeatedKFold(n_repeats=2, n_splits=10, random_state=42)
SMT
BG
outer
Fold: 0
SMT
('balance', 3)
Inner loop


r_index: 100%|##########| 377/377 [00:00<00:00, 602.27it/s]


Inner loop


r_index: 100%|##########| 400/400 [00:00<00:00, 910.13it/s]


Average SERA: 1100.1712773170339
('balance', 3)
SMT
('balance', 5)
Inner loop


...

synth_matrix: 100%|##########| 112/112 [00:02<00:00, 44.90it/s]


Average SERA: 979.7723571393494
('extreme', 5)


r_index: 100%|##########| 66/66 [00:00<00:00, 631.62it/s]


sera_outer 160.3371206043411


OSError: Cannot save file into a non-existent directory: '/Content/SMT/wine.csv/BaggingRegressor'